# 02 Can neighbours predict missing hours?
Hide each known area's hours, copy them from the nearest known area, measure accuracy by distance.

In [ ]:
from pathlib import Path
import geopandas as gpd, pandas as pd, matplotlib.pyplot as plt

RAW = sorted(Path("../data/raw").iterdir())[-1]  # latest snapshot
print("snapshot:", RAW.name)
areas = gpd.read_file(RAW / "parking_areas_3879.geojson")

In [ ]:
areas["hours"] = areas["voimassaolo"].fillna("").str.replace(r"\s+", "", regex=True)
known = areas[areas["hours"] != ""].reset_index(drop=True)
print(len(known), "known,", (areas["hours"] == "").sum(), "missing")

In [ ]:
def nearest(row, pool, same_class):
    cand = pool[pool.index != row.name]
    if same_class:
        cand = cand[cand["luokka"] == row["luokka"]]
    if cand.empty:
        return None, None
    d = cand.distance(row.geometry)
    return d.min(), cand.loc[d.idxmin(), "hours"]

sample = known.sample(500, random_state=0)
rows = []
for _, r in sample.iterrows():
    for same in (False, True):
        dist, h = nearest(r, known, same)
        rows.append({"same_class": same, "dist": dist, "match": h == r["hours"]})
res = pd.DataFrame(rows).dropna()
res.groupby("same_class")["match"].mean()

In [ ]:
res["dist_bin"] = pd.cut(res["dist"], [0, 10, 50, 200, 1e9])
res.groupby(["same_class", "dist_bin"], observed=True)["match"].agg(["mean", "size"])

## How far are the missing areas from a known same-class area?
If far, the test above is optimistic.

In [ ]:
missing = areas[(areas["hours"] == "") & areas["luokka"].isin([1, 5, 6, 7, 8, 9, 10])]
d = [known[known["luokka"] == r["luokka"]].distance(r.geometry).min() for _, r in missing.iterrows()]
pd.Series(d).describe()